In [38]:
from ingest import load_faq_data
documents = load_faq_data(file_path="../documents/all_documents.json") # adjust path as needed

In [39]:
documents[10]

{'course': 'data-engineering',
 'section': 'General Course-Related Questions',
 'question': 'Office Hours: I can’t attend the “Office hours” / workshop, will it be recorded?',
 'answer': 'Yes! Every "Office Hours" will be recorded and available a few minutes after the live session is over; so you can view (or rewatch) whenever you want.',
 'doc_id': 'a411de5004'}

In [40]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm":
        documents_llm.append(doc)

len(documents_llm)

111

In [41]:
documents = documents_llm

In [43]:
doc = documents[2]
print(doc["doc_id"])
print(doc["question"])
print(doc["answer"])

489dd1c9d9
What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/de) before it begins. You can also watch live on the SiteClub [YouTube Channel](https://www.youtube.com/c/SiteClub).

Don’t post questions in chat as they may be missed if the room is very active.


In [8]:
doc

{'course': 'llm',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [44]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [45]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [46]:
data_gen_instructions

"You emulate a student who's taking our course.\nFormulate 5 questions this student might ask based on a FAQ record. The record\nshould contain the answer to the questions, and the questions should be complete and not too short.\nIf possible, use as fewer words as possible from the record.\n\nThe output should resemble how people ask questions\non the internet. Not too formal, not too short, not too long."

In [47]:
from openai import OpenAI

openai_client = OpenAI(
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
    api_key="sk-ws-H.DDLHEXI.SR82.MEYCIQCBSGAps6xCs3yyu7H1VMvOUNsD9FfYYx5bT-bIK7wfGgIhAL6jYaXGbSFx4q2WkO5cF4SwilWUnlmrSU3Jknvw8LKK"
)

In [48]:
import json
user_prompt = json.dumps(doc)
user_prompt

'{"course": "llm", "section": "General Course-Related Questions", "question": "What is the video/zoom link to the stream for the \\u201cOffice Hours\\u201d or live/workshop sessions?", "answer": "The zoom link is only published to instructors/presenters/TAs.\\n\\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/de) before it begins. You can also watch live on the SiteClub [YouTube Channel](https://www.youtube.com/c/SiteClub).\\n\\nDon\\u2019t post questions in chat as they may be missed if the room is very active.", "doc_id": "489dd1c9d9"}'

In [49]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [50]:
response = openai_client.responses.create(
    model="qwen3.5-flash",
    # model="qwen-max",  # <--- Change this
    input=messages,
    # text_format=Questions
)

In [52]:
response

Response(id='resp_1631df37-1103-9b7c-b410-c95b29fc9a8a', created_at=1788681382.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='qwen3.5-flash', object='response', output=[ResponseReasoningItem(id='msg_77baa1a5-cede-4cdc-b59f-bcbbbd6cf07f', summary=[Summary(text='Thinking Process:\n\n1.  **Analyze the Request:**\n    *   **Role:** Emulate a student taking the course ("llm").\n    *   **Task:** Formulate 5 questions based on the provided FAQ record.\n    *   **Constraints:**\n        *   Questions must be completable answered by the record (though the prompt says "The record should contain the answer to the questions", it also says "use as fewer words as possible from the record" when formulating questions? No, it says "use as fewer words as possible from the record" likely meaning the questions shouldn\'t just copy-paste the text, they should sound natural. Wait, re-reading: "If possible, use as few words as possible from the record." This applies to the fo

In [53]:
doc

{'course': 'llm',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/de) before it begins. You can also watch live on the SiteClub [YouTube Channel](https://www.youtube.com/c/SiteClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'doc_id': '489dd1c9d9'}

In [17]:
from evaluation_utils import llm_structured

In [ ]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)


['Where can I find the YouTube Live link for the Office Hours or live sessions?', "How do I submit questions during the live sessions, and where's the Slido link?", 'Can I get the video URL for the live sessions before they start, and where is it posted?', 'What should I do if I miss the live session, can I watch it on a specific YouTube channel?', 'Is it okay to post my questions in the chat during the live session, or is there a better way?']


In [55]:
for i in (result.questions):
    print(i)

Where can I find the YouTube Live link for the Office Hours or live sessions?
How do I submit questions during the live sessions, and where's the Slido link?
Can I get the video URL for the live sessions before they start, and where is it posted?
What should I do if I miss the live session, can I watch it on a specific YouTube channel?
Is it okay to post my questions in the chat during the live session, or is there a better way?


In [56]:
usage

CompletionUsage(completion_tokens=116, prompt_tokens=349, total_tokens=465, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0))

In [57]:
from evaluation_utils import calc_price

In [58]:
calc_price(usage)

{'input_cost': 3.49e-05, 'output_cost': 4.64e-05, 'total_cost': 8.13e-05}

In [22]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["doc_id"]
    })

records

[{'question': 'Is it too late for me to enroll in the LLM course and still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I join the LLM course now and be eligible for a certificate if I complete the project?',
  'document': '74eb249bbf'},
 {'question': 'If I sign up for the LLM course today, can I still submit my project for certification?',
  'document': '74eb249bbf'},
 {'question': "I'm interested in the LLM course. Can I still join and get certified as long as I turn in my project on time?",
  'document': '74eb249bbf'},
 {'question': "What's the deadline for submitting projects in the LLM course to be considered for a certificate?",
  'document': '74eb249bbf'}]

In [23]:
import pandas as pd

In [24]:
pd.DataFrame(records)

,question,document
0,Is it too late for me to enroll in the LLM cou...,74eb249bbf
1,Can I join the LLM course now and be eligible ...,74eb249bbf
2,"If I sign up for the LLM course today, can I s...",74eb249bbf
3,I'm interested in the LLM course. Can I still ...,74eb249bbf
4,What's the deadline for submitting projects in...,74eb249bbf


In [25]:
from evaluation_utils import llm_structured_retry

In [26]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["doc_id"]
        })

    return results, usage

In [28]:
doc

{'course': 'llm',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [29]:
generate_ground_truth(doc)

([{'question': 'I stumbled across this class late, is enrollment still open for me?',
   'document': '74eb249bbf'},
  {'question': 'If I sign up today, will I qualify for the completion certificate later?',
   'document': '74eb249bbf'},
  {'question': 'Does the submission window stay the same for people who join after the start?',
   'document': '74eb249bbf'},
  {'question': "I'm worried the deadline for my final project might pass before I finish.",
   'document': '74eb249bbf'},
  {'question': "Can I actually get the credential if I'm submitting after everyone else starts?",
   'document': '74eb249bbf'}],
 CompletionUsage(completion_tokens=3143, prompt_tokens=252, total_tokens=3395, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=3038, rejected_prediction_tokens=None, text_tokens=3143), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=None, text_tokens=252)))

In [30]:
documents[:5]

[{'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM . When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
  'doc_id': '977bf7786c'},
 {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The zoom link is only published to instructors/

In [31]:
data_gen_instructions

"You emulate a student who's taking our course.\nFormulate 5 questions this student might ask based on a FAQ record. The record\nshould contain the answer to the questions, and the questions should be complete and not too short.\nIf possible, use as fewer words as possible from the record.\n\nThe output should resemble how people ask questions\non the internet. Not too formal, not too short, not too long."

In [33]:
len(documents[:5])

5

In [34]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
ground_truth

In [ ]:
len(ground_truth)
# pd.DataFrame(ground_truth)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
ground_truth[10]

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price
calc_total_price(usages)

In [36]:
df_ground_truth = pd.DataFrame(ground_truth)

In [37]:
df_ground_truth

,question,document
0,"Hi! I realized I signed up late for the class,...",74eb249bbf
1,"If I enroll after the start date, will I quali...",74eb249bbf
2,Do latecomers face different restrictions rega...,74eb249bbf
3,What is the deal with finishing the coursework...,74eb249bbf
4,Should I worry about missing out on the offici...,74eb249bbf
5,"Hi, I submitted my info already so when do I g...",977bf7786c
6,Is it safe to tackle the assignments now inste...,977bf7786c
7,Does providing details guarantee my entry into...,977bf7786c
8,"If we just show up anyway, what was the reason...",977bf7786c
9,Do you compare my work against a roster or jus...,977bf7786c


In [ ]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [ ]:
len(df_ground_truth)